# AethelVision™ Quickstart: Universal Sovereign Edge AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/aethelvision/blob/main/notebooks/Aethel_Quickstart.ipynb)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

Welcome to **AethelVision** — the clean-room, Apache-2.0 licensed, hardware-agnostic computer vision framework natively supporting:
1. **2D Object Detection** (Decoupled Anchor-Free + CIoU / DFL)
2. **Human Pose Estimation** (17 COCO Keypoints + OKS Loss)
3. **Instance Segmentation** (High-Speed ProtoNet on $P_3$)
4. **Oriented Bounding Boxes** (Continuous Angle $(\cos 2\theta, \sin 2\theta)$ + 2D Gaussian ProbIoU)

## 1. Environment Setup & Installation

In [ ]:
# Install dependencies and package in editable mode
!pip install -q torch torchvision numpy pillow pyyaml matplotlib
# When running in Colab:
# !git clone https://github.com/your-org/aethelvision.git
# %cd aethelvision
# !pip install -e .

## 2. Universal Hardware Discovery (HAL)

In [ ]:
from aethel.engine.hal import HardwareHAL

hw = HardwareHAL.discover_hardware(force_refresh=True)
print(f"Optimal Hardware Acceleration Backend: {hw['optimal_backend'].upper()}")
print(f"Detected Acceleration Backends:       {hw['detected_backends']}")

## 3. Instantiate Multi-Modal Neural Network Architectures

In [ ]:
import torch
from aethel.models import build_model

# 1. Object Detection (AethelNet-Small)
m_det = build_model("aethelnet_small", num_classes=80)
print("✓ Built AethelNet-Small Detector:", type(m_det))

# 2. Human Pose Estimation (17 COCO Keypoints)
m_pose = build_model("aethel_pose_small", num_classes=1, num_keypoints=17)
print("✓ Built AethelPose-Small Estimator:", type(m_pose))

# 3. Instance Segmentation (32 Prototypes)
m_seg = build_model("aethel_segment_small", num_classes=80, num_protos=32)
print("✓ Built AethelSegment-Small Segmenter:", type(m_seg))

# 4. Oriented Bounding Boxes (Continuous Trigonometric Angle Regression)
m_obb = build_model("aethel_obb_small", num_classes=15)
print("✓ Built AethelOBB-Small Detector:", type(m_obb))

## 4. Run Multi-Modal Inference & Visual Overlay Rendering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aethel.inference.visualizer import AethelVisualizer
from aethel.inference.result import PredictionResult
from aethel.models.head.obb import obb_to_corners

vis = AethelVisualizer()
canvas = np.zeros((400, 500, 3), dtype=np.uint8)
canvas[:] = (25, 30, 45)

# Construct demonstration predictions for 17-keypoint human pose
kpts = np.zeros((1, 17, 3), dtype=np.float32)
kpts[0, 0] = [250, 70, 0.95]   # Nose
kpts[0, 5] = [200, 130, 0.95]  # Left shoulder
kpts[0, 6] = [300, 130, 0.95]  # Right shoulder
kpts[0, 7] = [170, 190, 0.92]  # Left elbow
kpts[0, 8] = [330, 190, 0.92]  # Right elbow
kpts[0, 9] = [150, 250, 0.90]  # Left wrist
kpts[0, 10] = [350, 250, 0.90] # Right wrist
kpts[0, 11] = [210, 260, 0.95] # Left hip
kpts[0, 12] = [290, 260, 0.95] # Right hip
kpts[0, 13] = [200, 330, 0.90] # Left knee
kpts[0, 14] = [300, 330, 0.90] # Right knee
kpts[0, 15] = [190, 390, 0.85] # Left ankle
kpts[0, 16] = [310, 390, 0.85] # Right ankle

res_pose = PredictionResult(
    boxes=np.array([[140.0, 50.0, 360.0, 400.0]], dtype=np.float32),
    scores=np.array([0.96], dtype=np.float32),
    classes=np.array([0], dtype=np.int64),
    labels=["person"],
    keypoints=kpts,
)

rendered_img = vis.draw(canvas, res_pose)
plt.figure(figsize=(8, 6))
plt.imshow(rendered_img)
plt.title("AethelVision Real-Time 17-Keypoint Pose Estimation", color="white", fontsize=14)
plt.axis("off")
plt.show()

## 5. Universal Hardware Model Export

In [ ]:
from aethel.export.universal import UniversalExporter

exporter = UniversalExporter(model=m_det, opset_version=17)

# Export to Universal ONNX (Opset 17)
res_onnx = exporter.export("onnx", "exports/aethelnet_small.onnx")
print("ONNX Export Result:    ", res_onnx["status"])

# Export to Intel OpenVINO
res_ov = exporter.export("openvino", "exports/aethelnet_openvino/model.xml")
print("OpenVINO Export Result:", res_ov["status"])